In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DateType, TimestampType, FloatType

catalog_name = 'ecommerce'

In [0]:
df_bronze_brands = spark.table(f"{catalog_name}.bronze.brz_brands")
df_bronze_brands.show(10)

In [0]:
# 1.1 transformation: brz_brands - brand_code

df_silver_brands = df_bronze_brands.withColumn("brand_code", F.upper(F.regexp_replace(F.col("brand_code"), r'[^A-Za-z0-9]', '')))

df_silver_brands = df_silver_brands.dropDuplicates(["brand_code"])

df_silver_brands.show(100)

In [0]:
# 1.2 Validation: brz_brands - brand_code

df_silver_brands.groupBy("brand_code") \
.count() \
.filter(F.col("count") > 1) \
.show()

In [0]:
# 2.1 transformation: brz_brands - brand_name

df_silver_brands = df_silver_brands.withColumn("brand_name", F.trim(F.col("brand_name")))
df_silver_brands.show(100)

In [0]:
# 2.2 Validation: brz_brands - brand_name

df_silver_brands.groupBy("brand_name") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

In [0]:
# 3.1 transformation: brz_brands - category_code

df_silver_brands = df_silver_brands.withColumn("category_code", F.upper(F.col("category_code")))

category_anomalies = {
    "GROCERY": "GRCY",
    "BOOKS": "BKS",
    "TOYS": "TOY"
}

df_silver_brands = df_silver_brands.replace(category_anomalies, subset = "category_code")

valid_categories = ['CE','APP','AUTO','BPC','HLTH','GRCY','BKS','TOY','SPT','HNK']

df_silver_brands_clean = df_silver_brands.filter(F.col("category_code").isin(valid_categories))

df_silver_brands_quarantine = df_silver_brands.filter(~F.col("category_code").isin(valid_categories)) \
    .withColumn("rejection_reason", F.lit("invalid category_code"))

df_silver_brands_clean.show(100)

In [0]:
# 3.2 Validation: brz_brands - category_code

valid_categories = ['CE','APP','AUTO','BPC','HLTH','GRCY','BKS','TOY','SPT','HNK']
df_silver_brands_clean.filter(~F.col("category_code").isin(valid_categories)) \
    .select("category_code") \
    .show()

df_silver_brands_quarantine.select("category_code") \
.show()

In [0]:
df_silver_brands_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog_name}.silver.slv_brands_clean")

df_silver_brands_quarantine.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog_name}.silver.slv_brands_quarantine")